# ΔR/R₀ vs Bending Angle

Reads all sensor CSV files (excluding `50kohm-1.csv`), splits each recording into
13 equal segments → angles [0,20,40,60,80,100,120,100,80,60,40,20,0]°,
computes ΔR/R₀ and plots mean ± std.

Two plots are generated:
- **A** — up-sweep only (0 → 120°), matches publication style
- **B** — both sweeps (shows hysteresis)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# ── Config ────────────────────────────────────────────────────────────────────
FOLDER = Path(r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\Wearables-Glove-Project\Angulos")

FILES = [
    "50kohm-2.csv",
    "50kohm-3.csv",
    "150kohm.csv",
    "210kohm.csv",
    "250kohm.csv",
    "460kohm.csv",
]  # 50kohm-1 excluded (timing mismatch)

ANGLES  = [0, 20, 40, 60, 80, 100, 120, 100, 80, 60, 40, 20, 0]  # full protocol
N_SEG   = len(ANGLES)
UP_IDX  = list(range(7))    # segments 0-6  → angles 0→120
DN_IDX  = list(range(6,13)) # segments 6-12 → angles 120→0
TRIM    = 0.20   # skip first/last 20% of each segment (transition avoidance)
SMOOTH  = 50     # moving-average window (samples)

# ── Per-file extraction ───────────────────────────────────────────────────────
up_vals = defaultdict(list)   # angle -> [dR/R0 per file]
dn_vals = defaultdict(list)

for fname in FILES:
    df   = pd.read_csv(FOLDER / fname)
    vals = df.iloc[:, 1].rolling(SMOOTH, center=True, min_periods=1).mean().values
    n    = len(vals)
    seg  = n / N_SEG

    # R0: stable mean of the first 0° segment (middle 60%)
    pad0 = int(seg * TRIM)
    R0   = np.mean(vals[pad0 : int(seg * (1 - TRIM))])

    for idx_list, store in [(UP_IDX, up_vals), (DN_IDX, dn_vals)]:
        for i in idx_list:
            s   = int(i * seg)
            e   = int((i + 1) * seg)
            pad = int((e - s) * TRIM)
            m   = np.mean(vals[s + pad : e - pad])
            store[ANGLES[i]].append((m - R0) / R0)

    print(f"{fname:15s}  R0 = {R0:,.1f}")

# ── Statistics ────────────────────────────────────────────────────────────────
angles_up = [0, 20, 40, 60, 80, 100, 120]
means_up  = np.array([np.mean(up_vals[a])         for a in angles_up])
stds_up   = np.array([np.std(up_vals[a],  ddof=1) for a in angles_up])

angles_dn = [120, 100, 80, 60, 40, 20, 0]
means_dn  = np.array([np.mean(dn_vals[a])         for a in angles_dn])
stds_dn   = np.array([np.std(dn_vals[a],  ddof=1) for a in angles_dn])

print("\nUp sweep (0 → 120°):")
for a, m, s in zip(angles_up, means_up, stds_up):
    print(f"  {a:3d}°   {m:.4f} ± {s:.4f}")

print("\nDown sweep (120 → 0°):")
for a, m, s in zip(angles_dn, means_dn, stds_dn):
    print(f"  {a:3d}°   {m:.4f} ± {s:.4f}")


In [ ]:
# ── Plot A: up-sweep only (publication style) ─────────────────────────────────
fig, ax = plt.subplots(figsize=(5.5, 4.5))

ax.errorbar(
    angles_up, means_up, yerr=stds_up,
    fmt='o-',
    color='red',
    markerfacecolor='red',
    markeredgecolor='darkred',
    markersize=7,
    linewidth=1.4,
    elinewidth=1.2,
    capsize=5,
    capthick=1.2,
    label=f'Bending (n={len(FILES)})'
)

ax.set_xlabel('Bending Angle (°)', fontsize=13)
ax.set_ylabel(r'$\Delta R/R_0$',   fontsize=13)
ax.set_xlim(-5, 130)
ax.set_xticks(angles_up)
ax.set_ylim(bottom=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(direction='in', length=5, labelsize=11)
ax.legend(fontsize=10, frameon=False, loc='upper left')

fig.tight_layout()
path_a = Path("dR_R0_upsweep.png")
fig.savefig(path_a, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {path_a.resolve()}")


In [ ]:
# ── Plot B: both sweeps — shows hysteresis ────────────────────────────────────
fig, ax = plt.subplots(figsize=(5.5, 4.5))

# Up sweep
ax.errorbar(
    angles_up, means_up, yerr=stds_up,
    fmt='o-', color='red',
    markerfacecolor='red', markeredgecolor='darkred',
    markersize=7, linewidth=1.4, elinewidth=1.2, capsize=5, capthick=1.2,
    label='Loading (0 → 120°)'
)

# Down sweep (plot x-axis in decreasing order so curve goes right→left)
ax.errorbar(
    angles_dn, means_dn, yerr=stds_dn,
    fmt='s--', color='steelblue',
    markerfacecolor='steelblue', markeredgecolor='navy',
    markersize=7, linewidth=1.4, elinewidth=1.2, capsize=5, capthick=1.2,
    label='Unloading (120 → 0°)'
)

ax.set_xlabel('Bending Angle (°)', fontsize=13)
ax.set_ylabel(r'$\Delta R/R_0$',   fontsize=13)
ax.set_xlim(-5, 130)
ax.set_xticks([0, 20, 40, 60, 80, 100, 120])
ax.set_ylim(bottom=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(direction='in', length=5, labelsize=11)
ax.legend(fontsize=10, frameon=False, loc='upper left')

fig.tight_layout()
path_b = Path("dR_R0_hysteresis.png")
fig.savefig(path_b, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {path_b.resolve()}")
